# Chapter 9 — XVA, and the Exposure Engine Behind It

This one goes past reproducing the chapter. Part 1 rebuilds every number in it;
Part 2 onward **builds the exposure profile from a term-structure model** instead
of taking the chapter's stylized curve as given, and then does the things a real
XVA desk does with it.

| Part | What it does |
|---|---|
| 1 | Examples 9.1–9.6 and Table 9.1, on the chapter's own profile |
| 2 | A Hull–White engine: simulate the swap, get EE and PFE from paths |
| 3 | Collateral — threshold, minimum transfer amount, margin period of risk |
| 4 | Netting, and what a netting set is worth |
| 5 | Wrong-way risk, with an intensity model $\lambda(t)=e^{a+bX(t)}$ |
| 6 | The whole stack rebuilt on the simulated profile |

The chapter's profile is $2.904\sqrt{t}\,(5-t)/5$ — diffusion against
amortization, written as a smooth curve. **A simulated profile is not smooth**,
and Part 2 shows what the smoothing hides.

NumPy and SciPy.

In [1]:
import numpy as np
from math import exp, sqrt
from scipy.optimize import brentq

NOTIONAL, T = 100.0, 5.0            # $m, years
SPREAD, RECOVERY = 0.0120, 0.40     # counterparty CDS spread, recovery
LAMBDA = SPREAD / (1 - RECOVERY)    # the credit triangle: 2% a year
R_DISC, FUNDING, HURDLE = 0.04, 0.0060, 0.10


def show(label, got, printed, tol=6e-3):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %14.4f   book %12.4f   %s" % (label, got, printed, flag))


def EE_chapter(t):
    """The chapter's stylized profile in $m: sqrt(t) diffusion, (T-t) amortization."""
    return 2.904 * np.sqrt(t) * (T - t) / T


DURATION = (1 - exp(-R_DISC * T)) / R_DISC
show("discount-weighted duration", DURATION, 4.53)

discount-weighted duration                             4.5317   book       4.5300   ok


## 1. The chapter's numbers — Examples 9.1 to 9.6

Expected exposure prices; potential future exposure sets limits. They answer
different questions, which is why a credit officer quoting \$10.3m and a trader
quoting \$2.5m are not disagreeing.

In [2]:
peak_t = 5.0 / 3.0
sigma = 2.5 / 0.3989
show("peak expected exposure, $m", EE_chapter(peak_t), 2.5)
show("implied sigma, $m", sigma, 6.27)
show("PFE at 95%, $m", 1.645 * sigma, 10.3)
show("PFE / EE", 1.645 * sigma / 2.5, 4.12, tol=2e-2)

peak expected exposure, $m                             2.4994   book       2.5000   ok
implied sigma, $m                                      6.2672   book       6.2700   ok
PFE at 95%, $m                                        10.3096   book      10.3000   ok
PFE / EE                                               4.1238   book       4.1200   ok


In [3]:
mid = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
ee = EE_chapter(mid)
q = np.array([exp(-LAMBDA * (i - 1)) - exp(-LAMBDA * i) for i in range(1, 6)])
cover = (1 - RECOVERY) * q            # insuring $1 of exposure for the year
df = np.exp(-R_DISC * mid)

print("%6s %12s %16s %18s %16s" % ("year", "EE ($m)", "cost of cover", "discount factor",
                                   "contribution"))
for i in range(5):
    print("%6d %12.3f %15.3f%% %18.4f %16.5f" % (i + 1, ee[i], 100 * cover[i], df[i],
                                                 ee[i] * cover[i] * df[i]))
CVA = float((ee * cover * df).sum())
print("%6s %12s %16s %18s %16.5f" % ("", "", "", "CVA", CVA))

show("CVA, $", 1e6 * CVA, 94_000)
show("CVA, bp of notional", 1e4 * CVA / NOTIONAL, 9.4)
# Chapter 24's reading -- loss given default times default probability -- must agree
show("CVA as (1-R) x sum EE q DF, $", 1e6 * (1 - RECOVERY) * float((ee * q * df).sum()), 94_000)

  year      EE ($m)    cost of cover    discount factor     contribution
     1        1.848           1.188%             0.9802          0.02152
     2        2.490           1.165%             0.9418          0.02731
     3        2.296           1.141%             0.9048          0.02371
     4        1.630           1.119%             0.8694          0.01585
     5        0.616           1.097%             0.8353          0.00564
                                                    CVA          0.09404
CVA, $                                             94037.0293   book   94000.0000   ok
CVA, bp of notional                                    9.4037   book       9.4000   ok
CVA as (1-R) x sum EE q DF, $                      94037.0293   book   94000.0000   ok


In [4]:
epe_discounted = float((ee * df).sum())
show("discounted expected exposure, $m-years", epe_discounted, 8.17)
FVA = FUNDING * epe_discounted
show("FVA, $", 1e6 * FVA, 49_000, tol=1e-2)

MVA = FUNDING * 1.25 * DURATION                   # average initial margin of $1.25m
show("MVA, $", 1e6 * MVA, 34_000)

THRESH_EXPOSURE = 1.4                             # average uncollateralized amount left
show("CVA on a $5m threshold, $", 1e6 * (1 - RECOVERY) * LAMBDA * THRESH_EXPOSURE * DURATION,
     76_000)
show("funding on the threshold, $", 1e6 * FUNDING * THRESH_EXPOSURE * DURATION, 38_000)
show("total cost of the threshold, $",
     1e6 * ((1 - RECOVERY) * LAMBDA + FUNDING) * THRESH_EXPOSURE * DURATION, 114_000)

KVA = HURDLE * 0.4 * DURATION                     # $0.4m average regulatory capital
show("KVA, $", 1e6 * KVA, 181_000)

discounted expected exposure, $m-years                 8.1650   book       8.1700   ok
FVA, $                                             48990.0148   book   49000.0000   ok
MVA, $                                             33987.9838   book   34000.0000   ok
CVA on a $5m threshold, $                          76133.0837   book   76000.0000   ok
funding on the threshold, $                        38066.5419   book   38000.0000   ok
total cost of the threshold, $                    114199.6256   book  114000.0000   ok
KVA, $                                            181269.2469   book  181000.0000   ok


## Table 9.1, and what the DVA figure implies

The chapter's table gives DVA as $+39{,}000$ at a dealer spread it states as 49 basis
points. That spread can be recovered from the DVA alone, which checks the two against
each other: DVA is CVA computed on the dealer's own hazard against negative exposure,
and for an at-market swap the negative exposure profile equals the positive one by
symmetry.

In [5]:
def dva_at(hazard):
    qo = np.array([exp(-hazard * (i - 1)) - exp(-hazard * i) for i in range(1, 6)])
    return (1 - RECOVERY) * float((ee * qo * df).sum())


own_hazard = brentq(lambda h: 1e6 * dva_at(h) - 39_000, 1e-6, 0.10)
DVA = dva_at(own_hazard)
print("the dealer's own hazard implied by DVA = +39,000 is %.3f%%," % (100 * own_hazard))
print("i.e. a CDS spread of about %.0f bp at %.0f%% recovery"
      % (1e4 * own_hazard * (1 - RECOVERY), 100 * RECOVERY))

print()
print("%-8s %18s %14s" % ("", "uncollateralized", "cleared"))
# cleared capital ~$28,000: Example 27.3's EAD 0.929m x 3% leverage ratio
rows = [("CVA", -CVA * 1e6, 0.0), ("DVA", DVA * 1e6, 0.0), ("FVA", -FVA * 1e6, 0.0),
        ("MVA", None, -MVA * 1e6), ("KVA", -KVA * 1e6, -HURDLE * 28_000 * DURATION)]
fmt = lambda v: "---" if v is None else "{:+,.0f}".format(v)
for name, u, c in rows:
    print("%-8s %18s %14s" % (name, fmt(u), fmt(c)))
tot_u = -CVA * 1e6 + DVA * 1e6 - FVA * 1e6 - KVA * 1e6
tot_c = -MVA * 1e6 - HURDLE * 28_000 * DURATION
print("%-8s %18s %14s" % ("total", fmt(tot_u), fmt(tot_c)))
show("uncollateralized total, $", tot_u, -285_000)
show("cleared total, $", tot_c, -47_000, tol=1e-2)

the dealer's own hazard implied by DVA = +39,000 is 0.809%,
i.e. a CDS spread of about 49 bp at 40% recovery

           uncollateralized        cleared
CVA                 -94,037             +0
DVA                 +39,000             +0
FVA                 -48,990             +0
MVA                     ---        -33,988
KVA                -181,269        -12,689
total              -285,296        -46,677
uncollateralized total, $                        -285296.2911   book -285000.0000   ok
cleared total, $                                  -46676.8311   book  -47000.0000   ok


## 2. Build the exposure instead of assuming it

A Hull–White short rate on a flat initial curve, a par five-year payer swap, and
40,000 antithetic paths. Two checks come first: the swap must be worth zero at
inception, and the simulated bond prices must reprice the initial curve.

**Then look at the shape.** The chapter's curve is smooth. The simulated one
sawtooths, because exposure drops the instant a coupon is paid — the profile is
a series of humps between payment dates, not one hump.

In [6]:
A_MR, SIG_R, R0 = 0.05, 0.010, 0.04          # mean reversion, short-rate vol, flat curve
PAY_TIMES = np.arange(0.5, T + 1e-9, 0.5)


def B(tau):
    return (1 - np.exp(-A_MR * tau)) / A_MR


def par_rate():
    annuity = sum(0.5 * exp(-R0 * t) for t in PAY_TIMES)
    return (1 - exp(-R0 * T)) / annuity


K_SWAP = par_rate()
print("par swap rate on the flat %.1f%% curve: %.4f%%" % (100 * R0, 100 * K_SWAP))


def simulate(n_paths=20_000, n_steps=60, seed=3):
    """Hull-White paths, and the payer-swap value on each date.

    Between reset dates the current floating payment is ALREADY FIXED, so the
    reset-date formula N[(1-P(t,T_n)) - K A(t)] is wrong there. The last fixing
    is carried per path and valued explicitly; without it the profile jumps the
    wrong way at every payment date.
    """
    dt = T / n_steps
    delta = 0.5
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, n_steps))
    z = np.concatenate([z, -z])                   # antithetic
    m = len(z)
    x = np.zeros(m)
    L_fixed = np.full(m, K_SWAP)                  # rate fixed at t=0 for the first period
    grid, values, factor = [], [], []

    def bond(t, u, x):
        Vt = SIG_R ** 2 * (1 - exp(-2 * A_MR * t)) / (2 * A_MR) if t > 0 else 0.0
        return np.exp(-R0 * (u - t) - B(u - t) * x - 0.5 * B(u - t) ** 2 * Vt)

    for i in range(n_steps):
        t_prev = i * dt
        x = x * exp(-A_MR * dt) + SIG_R * sqrt((1 - exp(-2 * A_MR * dt)) / (2 * A_MR)) * z[:, i]
        t = (i + 1) * dt
        # if a reset date was crossed, the new period's rate fixes there
        for tr in PAY_TIMES[:-1]:
            if t_prev < tr - 1e-9 <= t - 1e-9 or (t_prev < tr <= t):
                P_next = bond(tr, tr + delta, x)
                L_fixed = (1.0 / P_next - 1.0) / delta
        grid.append(t)
        factor.append(x.copy())
        future = PAY_TIMES[PAY_TIMES > t + 1e-9]
        if len(future) == 0:
            values.append(np.zeros(m))
            continue
        T_j, T_n = future[0], future[-1]
        P = np.array([bond(t, u, x) for u in future])
        floating = delta * L_fixed * P[0] + (P[0] - P[-1] if len(future) > 1 else 0.0)
        fixed = K_SWAP * delta * P.sum(axis=0)
        values.append(NOTIONAL * (floating - fixed))
    return np.array(grid), np.array(values), np.array(factor)


grid, V, X = simulate()
EE_mc = np.maximum(V, 0).mean(axis=1)
PFE_mc = np.quantile(np.maximum(V, 0), 0.95, axis=1)

print("value at the first step, mean %.4f  (a par swap starts at zero)" % V[0].mean())
print("peak EE %.3f at t = %.2f;  peak PFE %.3f" % (EE_mc.max(), grid[EE_mc.argmax()],
                                                    PFE_mc.max()))
print("PFE / EE at the peak: %.2f   (Example 9.1 gets %.2f from the normal approximation)"
      % (PFE_mc[EE_mc.argmax()] / EE_mc[EE_mc.argmax()], 10.3 / 2.5))

par swap rate on the flat 4.0% curve: 4.0403%


value at the first step, mean 0.0001  (a par swap starts at zero)


peak EE 1.595 at t = 1.92;  peak PFE 6.494
PFE / EE at the peak: 4.07   (Example 9.1 gets 4.12 from the normal approximation)


In [7]:
print("the simulated profile against the chapter's smooth curve, rescaled to the same peak")
print("%8s %14s %16s %14s" % ("t", "simulated", "chapter", "difference"))
scale = 2.5 / EE_mc.max()
for t in (0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5):
    j = int(np.argmin(abs(grid - t)))
    a, b = scale * EE_mc[j], EE_chapter(t)
    print("%8.2f %14.3f %16.3f %14.3f" % (t, a, b, a - b))
print()
print("the sawtooth: EE on the step before and the step after each payment date")
for t_pay in (1.0, 2.0, 3.0, 4.0):
    j1 = int(np.argmin(abs(grid - t_pay)))         # the payment has settled by this point
    j0 = j1 - 1                                    # the step before it
    print("  t=%.1f   before %.3f   after %.3f   drop %.1f%%"
          % (t_pay, scale * EE_mc[j0], scale * EE_mc[j1],
             100 * (1 - EE_mc[j1] / EE_mc[j0])))

the simulated profile against the chapter's smooth curve, rescaled to the same peak
       t      simulated          chapter     difference
    0.25          1.135            1.379         -0.245
    0.50          1.627            1.848         -0.221
    1.00          2.059            2.323         -0.264
    1.50          2.243            2.490         -0.247
    2.00          2.228            2.464         -0.236
    2.50          2.109            2.296         -0.187
    3.00          1.875            2.012         -0.137
    3.50          1.520            1.630         -0.109
    4.00          1.095            1.162         -0.066
    4.50          0.588            0.616         -0.028

the sawtooth: EE on the step before and the step after each payment date
  t=1.0   before 2.130   after 2.059   drop 3.3%
  t=2.0   before 2.500   after 2.228   drop 10.9%
  t=3.0   before 2.282   after 1.875   drop 17.8%
  t=4.0   before 1.597   after 1.095   drop 31.4%


## 3. Collateral — threshold, MTA, and the margin period of risk

A CSA does not remove exposure; it removes the part above the threshold, and only
after a delay. The **margin period of risk** is the window between the last
collateral call the counterparty honored and the close-out — typically ten days
uncleared. Exposure over that window is uncollateralized whatever the CSA says.

Example 9.5 prices a \$5m threshold at about \$114,000 of CVA and FVA together. Here it is computed from the
paths rather than assumed.

In [8]:
def collateralized_exposure(V, grid, threshold, mta=0.25, mpor_days=10):
    """Exposure after a CSA: collateral chases the value with a lag and a threshold."""
    mpor = mpor_days / 250.0
    lag = max(1, int(round(mpor / (grid[1] - grid[0]))))
    posted = np.zeros_like(V)
    for i in range(len(grid)):
        target = np.maximum(V[i] - threshold, 0.0)
        prev = posted[i - 1] if i else np.zeros(V.shape[1])
        move = np.where(np.abs(target - prev) >= mta, target - prev, 0.0)
        posted[i] = prev + move
    held = np.vstack([np.zeros((lag, V.shape[1])), posted[:-lag]])   # collateral is stale
    return np.maximum(V - held, 0.0)


def cva_from_paths(exposure, grid, hazard=LAMBDA, recovery=RECOVERY):
    ee_t = exposure.mean(axis=1)
    dt = grid[1] - grid[0]
    surv = np.exp(-hazard * grid)
    dq = np.concatenate([[1 - surv[0]], surv[:-1] - surv[1:]])
    return (1 - recovery) * float((ee_t * dq * np.exp(-R_DISC * grid)).sum())


base = np.maximum(V, 0.0)
print("%14s %16s %16s %14s" % ("threshold ($m)", "average EE", "CVA ($)", "vs no CSA"))
uncol = cva_from_paths(base, grid)
for th in (np.inf, 10.0, 5.0, 2.0, 0.0):
    ex = base if np.isinf(th) else collateralized_exposure(V, grid, th)
    c = cva_from_paths(ex, grid)
    label = "none" if np.isinf(th) else "%.0f" % th
    print("%14s %16.3f %16s %13.0f%%" % (label, ex.mean(axis=1).mean(),
                                         "{:,.0f}".format(1e6 * c), 100 * c / uncol))
print()
print("a zero threshold does not give zero CVA: the margin period of risk remains")

threshold ($m)       average EE          CVA ($)      vs no CSA
          none            1.094           57,458           100%
            10            1.093           57,401           100%
             5            1.025           53,801            94%
             2            0.684           35,842            62%
             0            0.126            6,917            12%

a zero threshold does not give zero CVA: the margin period of risk remains


## 4. Netting

Two trades with the same counterparty. Under a netting agreement exposure is
$\max(\sum V_i, 0)$; without one it is $\sum \max(V_i, 0)$, and the difference is
the whole reason ISDA close-out netting exists.

Here the second trade is the offsetting receiver swap — the extreme case, where
netting removes almost everything.

In [9]:
V2 = -0.7 * V                                  # an offsetting receiver swap, 70% of the size
gross = np.maximum(V, 0.0) + np.maximum(V2, 0.0)
netted = np.maximum(V + V2, 0.0)

cva_gross = cva_from_paths(gross, grid)
cva_net = cva_from_paths(netted, grid)
print("average exposure, gross  %.3f $m" % gross.mean(axis=1).mean())
print("average exposure, netted %.3f $m" % netted.mean(axis=1).mean())
print("CVA gross  {:,.0f}".format(1e6 * cva_gross))
print("CVA netted {:,.0f}".format(1e6 * cva_net))
print("netting benefit %.0f%%" % (100 * (1 - cva_net / cva_gross)))

average exposure, gross  1.858 $m
average exposure, netted 0.328 $m
CVA gross  97,611
CVA netted 17,237
netting benefit 82%


## 5. Wrong-way risk

Section 9.9 names intensity models, which let the counterparty's default rate rise
and fall with the driver of the exposure. The standard form is
$\lambda(t) = e^{a + bX(t)}$ with $b>0$. Applied here, $X$ is the Hull–White factor: a payer swap gains when rates
rise, so $b>0$ makes the counterparty likeliest to fail exactly when the dealer is
owed most.

$a$ is set at each date so the **marginal** default probability is unchanged. That
matters: without it, raising $b$ would also raise the overall default rate, and the
two effects would be confounded. Holding the marginal fixed isolates the
dependence, and what remains is the exposure conditional on default, which
Section 9.9 says the standard calculation replaces with the unconditional average.

In [10]:
def cva_wrong_way(V, X, grid, b, hazard=LAMBDA, recovery=RECOVERY):
    """lambda(t) = exp(a + b X(t)), with a chosen so E[lambda] = hazard at every date."""
    dt = grid[1] - grid[0]
    tilt = np.exp(b * X)
    tilt = tilt / tilt.mean(axis=1, keepdims=True)      # E[tilt] = 1, so the marginal is unchanged
    lam = hazard * tilt
    surv = np.exp(-np.cumsum(lam, axis=0) * dt)
    prev = np.vstack([np.ones((1, V.shape[1])), surv[:-1]])
    dq = prev - surv                                    # default probability in each step
    ex = np.maximum(V, 0.0)
    return (1 - recovery) * float((ex * dq * np.exp(-R_DISC * grid)[:, None]).mean(axis=1).sum())


independent = cva_wrong_way(V, X, grid, 0.0)
print("%10s %18s %16s" % ("b", "CVA ($)", "vs independent"))
for b in (-40.0, -20.0, 0.0, 20.0, 40.0, 80.0):
    c = cva_wrong_way(V, X, grid, b)
    tag = "right-way" if b < 0 else ("independent" if b == 0 else "wrong-way")
    print("%10.0f %18s %14.2fx   %s" % (b, "{:,.0f}".format(1e6 * c),
                                        c / independent, tag))
print()
print("independence is a lower bound in the wrong-way case, as Section 9.9 says")
print("the regulatory answer is a flat 1.4 multiplier (Section 27.5)")

         b            CVA ($)   vs independent
       -40             28,240           0.49x   right-way


       -20             40,954           0.71x   right-way


         0             57,458           1.00x   independent
        20             77,567           1.35x   wrong-way


        40            100,312           1.75x   wrong-way


        80            144,092           2.51x   wrong-way

independence is a lower bound in the wrong-way case, as Section 9.9 says
the regulatory answer is a flat 1.4 multiplier (Section 27.5)


## 6. The whole stack, on the simulated profile

Rebuild Table 9.1 from the paths rather than from the chapter's curve. The
numbers will not match to the dollar (the simulated profile has a different peak
and a sawtooth), but **the ordering must survive**, and it is the ordering that
carries the chapter's argument: KVA largest, then CVA, then FVA, then MVA; and under
clearing, MVA replacing CVA and FVA at a fraction of their cost.

In [11]:
ee_t = np.maximum(V, 0.0).mean(axis=1)
dt = grid[1] - grid[0]
disc = np.exp(-R_DISC * grid)
scale = 2.5 / ee_t.max()                        # put it on the chapter's scale to compare

cva_sim = scale * cva_from_paths(np.maximum(V, 0.0), grid)
fva_sim = FUNDING * scale * float((ee_t * disc * dt).sum())
dva_sim = scale * cva_from_paths(np.maximum(V, 0.0), grid, hazard=own_hazard)
kva_sim = HURDLE * 0.4 * DURATION
mva_sim = FUNDING * 1.25 * DURATION

print("%-8s %20s %20s" % ("", "simulated ($)", "chapter ($)"))
for name, sim, book in (("CVA", -cva_sim, -CVA), ("DVA", dva_sim, DVA),
                        ("FVA", -fva_sim, -FVA), ("KVA", -kva_sim, -KVA)):
    print("%-8s %20s %20s" % (name, "{:+,.0f}".format(1e6 * sim), "{:+,.0f}".format(1e6 * book)))
print("%-8s %20s %20s" % ("total",
                          "{:+,.0f}".format(1e6 * (-cva_sim + dva_sim - fva_sim - kva_sim)),
                          "{:+,.0f}".format(1e6 * (-CVA + DVA - FVA - KVA))))
print()
order = sorted([("CVA", cva_sim), ("FVA", fva_sim), ("KVA", kva_sim), ("MVA", mva_sim)],
               key=lambda p: -p[1])
print("largest to smallest: " + " > ".join(n for n, _ in order))

                simulated ($)          chapter ($)
CVA                   -90,063              -94,037
DVA                   +37,400              +39,000
FVA                   -47,021              -48,990
KVA                  -181,269             -181,269
total                -280,954             -285,296

largest to smallest: KVA > CVA > FVA > MVA
